# Sprint 4 — Sesión Teórica — Versión estudiante
## Del proceso de negocio al funnel: CTEs, conversión y retención con SQL

**Modalidad:** Coding Together  
**Duración:** 70 minutos  
**Motor:** DuckDB en Google Colab  
**Datos:** archivos `.parquet` con prefijo `sprint04_`

> **Hilo conductor:** negocio → proceso → etapas → CTEs → métricas → decisión.


## Cómo usar esta versión

Este notebook está diseñado para acompañar la explicación del tutor.

- Ejecuta completas las celdas de infraestructura.
- Completa las consultas marcadas como **TODO** durante el Coding Together.
- Antes de escribir SQL, responde qué representa una fila y qué pregunta de negocio resolverás.
- Registra resultados e interpretaciones en los espacios asignados.
- Las pistas orientan el procedimiento, pero no contienen la consulta final.


## Puente entre Sprint 3 y Sprint 4

En Sprint 3 aprendimos piezas de SQL: `SELECT`, `WHERE`, `GROUP BY`, `CASE`, `JOIN` y funciones de agregación.

En Sprint 4 las combinaremos para responder una pregunta de negocio completa:

```text
Problema de negocio
        ↓
Proceso del usuario
        ↓
Etapas medibles
        ↓
CTE por etapa
        ↓
Conteos y tasas
        ↓
Hallazgo y decisión
```

### Mis conocimientos previos

- Operación SQL que manejo con mayor seguridad: __________________________
- Operación SQL que necesito practicar: _________________________________
- Pregunta que quiero resolver durante la sesión: ________________________


## Objetivos de aprendizaje

Al finalizar la sesión podrás:

1. Reconocer qué representa una fila y la granularidad de `users` y `events`.
2. Traducir un proceso de compra en etapas ordenadas.
3. Construir una CTE independiente para cada etapa.
4. Unir las etapas con `LEFT JOIN` para contar usuarios que avanzan.
5. Calcular conversiones y drop-off con denominadores correctos.
6. Interpretar el cuello de botella y formular una conclusión de negocio.
7. Diferenciar una pregunta de conversión de una pregunta de retención.


## Metodología, reglas y entregable

1. Primero se formula la pregunta de negocio; después se escribe SQL.
2. Cada CTE representa una sola etapa.
3. Antes de avanzar, se valida qué representa una fila.
4. Todo resultado debe interpretarse con lenguaje de negocio.
5. Si una consulta falla, se lee el error antes de copiar una solución.
6. Solo realizaremos verificaciones mínimas; la sesión no se centrará en limpieza.

**Entregable de la sesión**

- esquema de tablas y llaves;
- funnel de cinco etapas con una CTE por etapa;
- conteos, conversiones y drop-off;
- análisis por país;
- tabla básica de retención por cohorte;
- conclusión con métrica, hallazgo y acción.


## Preguntas guiadas

1. ¿Qué decisión puede tomar una empresa al conocer dónde se pierden sus usuarios?
2. ¿Qué representa una fila en una tabla de eventos?
3. ¿Por qué el orden de las etapas importa?
4. ¿Qué debería conservar un `LEFT JOIN`: la etapa inicial o la final?
5. ¿Conversión y retención responden la misma pregunta?

### Mis respuestas iniciales

1.  
2.  
3.  
4.  
5.  

## Estructura y timeboxing

| Tiempo | Bloque | Pregunta central |
|---:|---|---|
| 0–5 | Apertura | ¿Qué problema de negocio resolveremos? |
| 5–12 | Bloque 0 | ¿Qué representa cada tabla y cada fila? |
| 12–22 | Bloque 1 | ¿Cómo se convierte un journey en etapas? |
| 22–40 | Bloque 2 | ¿Cómo ordenamos una CTE por etapa? |
| 40–53 | Bloque 3 | ¿Dónde está el cuello de botella? |
| 53–63 | Bloque 4 | ¿Qué cambia al analizar retención? |
| 63–70 | Cierre | ¿Cómo se aplica esto al proyecto? |


<div style="text-align: center">
    <img src="https://raw.githubusercontent.com/ljpiere/tpdata_python/main/images/w1s1_2.png" width="400">
</div>


---
# Bloque 0 · Contexto de negocio y modelo de datos

Un marketplace observa que los registros crecen, pero las compras no aumentan al mismo ritmo.

> **Pregunta de negocio:** ¿en qué etapa del proceso de compra se pierde la mayor proporción de usuarios y qué segmento requiere atención?

```text
users                                      events
────────────────────────                   ─────────────────────────
user_id          PK  ────────────────┐     event_id        PK
signup_ts                              └──< user_id         FK
segment                                   session_id
country                                   event_ts
plan                                      event_name
channel                                   platform
device                                    device, page
```

| Tabla | Una fila representa | Llave | Uso |
|---|---|---|---|
| `users` | un usuario | `user_id` | describir quién es |
| `events` | una acción | `event_id` | reconstruir qué hizo y cuándo |
| `funnel_sessions` | una sesión experimental | `session_id` | comparar A/B |
| `experiments` | una variante | `experiment_id`, `variant` | comprender el cambio |


## Preparación del entorno

Ejecuta estas celdas completas. Son infraestructura; el trabajo analítico comienza después.


In [ ]:
# Infraestructura de la clase: ejecutar una vez.
%pip install -q duckdb pandas pyarrow requests


In [ ]:
from pathlib import Path
import requests
import pandas as pd
import duckdb

BASE_URL = "https://raw.githubusercontent.com/ljpiere/tpdata_python/refs/heads/main/DA/DA_students/datasets/"

DATASETS = {
    "users": "sprint04_users.parquet",
    "events": "sprint04_events.parquet",
    "funnel_sessions": "sprint04_funnel_sessions.parquet",
    "experiments": "sprint04_experiments.parquet",
}

def read_parquet_source(file_name: str) -> pd.DataFrame:
    """Lee primero el archivo local; si no existe, lo descarga temporalmente."""
    source_path = Path(file_name)

    if not source_path.exists():
        response = requests.get(f"{BASE_URL}{file_name}", timeout=30)
        response.raise_for_status()

        source_path = Path("/tmp") / file_name
        source_path.write_bytes(response.content)

    # DuckDB evita incompatibilidades de metadatos entre versiones de PyArrow.
    reader = duckdb.connect(database=":memory:")
    try:
        return reader.execute(
            "SELECT * FROM read_parquet(?)",
            [str(source_path)],
        ).df()
    finally:
        reader.close()

users = read_parquet_source(DATASETS["users"])
events = read_parquet_source(DATASETS["events"])
funnel_sessions = read_parquet_source(DATASETS["funnel_sessions"])
experiments = read_parquet_source(DATASETS["experiments"])

con = duckdb.connect(database=":memory:")
con.register("users_df", users)
con.register("events_df", events)
con.register("funnel_sessions_df", funnel_sessions)
con.register("experiments_df", experiments)

con.execute("CREATE OR REPLACE TABLE users AS SELECT * FROM users_df")
con.execute("CREATE OR REPLACE TABLE events AS SELECT * FROM events_df")
con.execute("CREATE OR REPLACE TABLE funnel_sessions AS SELECT * FROM funnel_sessions_df")
con.execute("CREATE OR REPLACE TABLE experiments AS SELECT * FROM experiments_df")

def sql(query: str) -> pd.DataFrame:
    return con.execute(query).df()

print("Tablas listas:", ", ".join(DATASETS.keys()))


## 0.1 Verificación mínima de carga

Esta consulta únicamente comprueba que las cuatro tablas están disponibles. No es un ejercicio de limpieza.


In [ ]:
sql("""
SELECT 'users' AS table_name, COUNT(*) AS rows FROM users
UNION ALL
SELECT 'events', COUNT(*) FROM events
UNION ALL
SELECT 'funnel_sessions', COUNT(*) FROM funnel_sessions
UNION ALL
SELECT 'experiments', COUNT(*) FROM experiments;
""")


## 0.2 Reconocer la granularidad

Ejecuta las dos consultas y completa la ficha.

| Tabla | ¿Qué representa una fila? | Llave principal | Campo temporal |
|---|---|---|---|
| `users` |  |  |  |
| `events` |  |  |  |


In [ ]:
sql("DESCRIBE users")


In [ ]:
sql("DESCRIBE events")


### Validación de comprensión

- Una fila de `users` representa _______________________________________.
- Una fila de `events` representa ______________________________________.
- La relación entre ambas tablas se construye con _______________________.
- El orden temporal de los eventos se determina con ____________________.

### Decisión de unidad de análisis

Para el funnel de esta sesión contaremos:

- [ ] filas;
- [ ] sesiones;
- [ ] usuarios únicos.

**Justificación:** _____________________________________________________


---
# Bloque 1 · Del journey al funnel

Un **journey** describe el recorrido completo del usuario. Un **funnel** selecciona etapas clave para medir avance y pérdida.

```text
signup → view_product → add_to_cart → begin_checkout → purchase
```

Antes de crear CTEs, comprobaremos que esas categorías existen y observaremos el orden de algunos eventos.


In [ ]:
sql("""
SELECT
    user_id,
    session_id,
    event_ts,
    event_name
FROM events
ORDER BY user_id, event_ts
LIMIT 15;
""")


In [ ]:
sql("""
SELECT DISTINCT event_name
FROM events
ORDER BY event_name;
""")


## Ejercicio 1 · Traducir negocio a etapas

Completa la tabla antes de escribir SQL.

| Orden | Etapa | Evento | ¿Qué significa para el negocio? |
|---:|---|---|---|
| 1 | Registro | `signup` |  |
| 2 | Exploración | `view_product` |  |
| 3 | Intención | `add_to_cart` |  |
| 4 | Inicio de pago | `begin_checkout` |  |
| 5 | Resultado | `purchase` |  |

### Pregunta de validación

¿Por qué no basta con ejecutar `COUNT(*)` sobre `events`?

**Respuesta:** _________________________________________________________


---
# Bloque 2 · CTEs: una etapa, una responsabilidad

Una **CTE** es un resultado temporal con nombre. En este funnel, cada CTE debe contener los usuarios que alcanzaron una sola etapa.

## Ejercicio 2.1 · Construir el funnel de conteos

**Objetivo:** crear cinco CTEs, unirlas desde `signup` y contar cuántos usuarios alcanzaron cada etapa.

### Antes de escribir

- Unidad de análisis: __________________________
- Tabla base: __________________________________
- Llave de unión: ______________________________
- CTE que debe quedar a la izquierda: __________


In [ ]:
# TODO — Completa la consulta durante el Coding Together.
#
# Patrón esperado:
#
# sql("""
# WITH signup AS (
#     SELECT DISTINCT user_id
#     FROM events
#     WHERE event_name = 'signup'
# ),
# view_product AS (
#     -- TODO: usuarios que alcanzaron view_product
# ),
# add_to_cart AS (
#     -- TODO: usuarios que alcanzaron add_to_cart
# ),
# begin_checkout AS (
#     -- TODO: usuarios que alcanzaron begin_checkout
# ),
# purchase AS (
#     -- TODO: usuarios que alcanzaron purchase
# )
# SELECT
#     -- TODO: contar cada etapa
# FROM signup AS s
# LEFT JOIN view_product AS v ON ...
# LEFT JOIN add_to_cart AS c ON ...
# LEFT JOIN begin_checkout AS ch ON ...
# LEFT JOIN purchase AS p ON ...;
# """)
#
# Escribe tu solución debajo:


<details>
<summary><strong>Pistas del ejercicio 2.1</strong></summary>

1. Usa `SELECT DISTINCT user_id` en cada CTE.
2. Cambia únicamente el valor de `event_name` en cada etapa.
3. Comienza el `FROM` con la CTE `signup`.
4. Une todas las etapas mediante `user_id`.
5. Usa `COUNT(alias.user_id)` para obtener los conteos.

</details>

### Registro de resultados

| Etapa | Usuarios | ¿Disminuye frente a la anterior? |
|---|---:|---|
| `signup` |  |  |
| `view_product` |  |  |
| `add_to_cart` |  |  |
| `begin_checkout` |  |  |
| `purchase` |  |  |

### Interpretación preliminar

El mayor descenso en volumen parece ocurrir entre __________________ y __________________.


---
# Bloque 3 · De conteos a decisiones

- **Conversión acumulada:** etapa / inicio.
- **Conversión entre etapas:** etapa actual / etapa anterior.
- **Drop-off:** `100 % - conversión entre etapas`.

## Ejercicio 3.1 · Calcular tasas

Reutiliza las cinco CTEs del ejercicio anterior y agrega una CTE llamada `funnel_counts`.


In [ ]:
# TODO — Completa las tasas.
#
# Estructura orientadora:
#
# sql("""
# WITH signup AS (...),
# view_product AS (...),
# add_to_cart AS (...),
# begin_checkout AS (...),
# purchase AS (...),
# funnel_counts AS (
#     SELECT
#         -- conteos del funnel
#     FROM signup AS s
#     LEFT JOIN ...
# )
# SELECT
#     *,
#     ROUND(100.0 * users_view_product
#           / NULLIF(users_signup, 0), 1) AS signup_to_view_pct,
#     -- TODO: view_to_cart_pct
#     -- TODO: cart_to_checkout_pct
#     -- TODO: checkout_to_purchase_pct
#     -- TODO: total_conversion_pct
# FROM funnel_counts;
# """)
#
# Escribe tu solución debajo:


<details>
<summary><strong>Pistas del ejercicio 3.1</strong></summary>

- El denominador de una conversión entre etapas es la etapa inmediatamente anterior.
- El denominador de la conversión total es `users_signup`.
- Usa `NULLIF(denominador, 0)` para evitar división por cero.
- Redondea después de multiplicar por `100.0`.

</details>

## Mi lectura del funnel

```text
La mayor pérdida relativa ocurre entre __________________ y __________________.
En esa transición avanza aproximadamente el ________ %.
Esto sugiere revisar _________________________________________________.
```


## Ejercicio 3.2 · Extender el funnel por país

**Pregunta:** ¿la conversión total es similar entre países?

Añade `country` desde `users`, conserva una CTE por etapa y agrupa el resultado final.


In [ ]:
# TODO — Funnel por país.
#
# Pasos:
# 1. Une `events` con `users` dentro de la CTE `signup`.
# 2. Selecciona `user_id` y `country`.
# 3. Conserva las demás CTEs por `user_id`.
# 4. Agrupa por `s.country`.
# 5. Calcula purchase / signup.
#
# Escribe tu consulta:


### Interpretación por país

| País | Conversión total | Volumen de registros | Hipótesis, no conclusión causal |
|---|---:|---:|---|
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

**Información adicional necesaria antes de recomendar una acción:**  
______________________________________________________________________


---
# Bloque 4 · Funnel y retención no responden lo mismo

| Análisis | Pregunta |
|---|---|
| Funnel | ¿Hasta qué etapa avanzan? |
| Retención | ¿Regresan o siguen activos? |
| Cohorte | ¿Cómo se comporta un grupo registrado en un periodo común? |

## Ejercicio 4.1 · Retención semanal por cohorte mensual

**Objetivo:** agrupar usuarios por mes de registro y calcular qué porcentaje abrió la aplicación después de una, dos y tres semanas.


In [ ]:
# TODO — Completa la consulta de retención.
#
# sql("""
# WITH activity AS (
#     SELECT
#         u.user_id,
#         DATE_TRUNC('month', u.signup_ts) AS cohort_month,
#         DATE_DIFF(
#             'week',
#             CAST(u.signup_ts AS DATE),
#             CAST(e.event_ts AS DATE)
#         ) AS week_after_signup,
#         e.event_name
#     FROM users AS u
#     LEFT JOIN events AS e ON u.user_id = e.user_id
# ),
# cohort_retention AS (
#     SELECT
#         cohort_month,
#         COUNT(DISTINCT user_id) AS cohort_size,
#         -- TODO: usuarios activos en semanas 1, 2 y 3
#     FROM activity
#     GROUP BY cohort_month
# )
# SELECT
#     cohort_month,
#     cohort_size,
#     -- TODO: conteos y porcentajes
# FROM cohort_retention
# ORDER BY cohort_month;
# """)
#
# Escribe tu solución debajo:


<details>
<summary><strong>Pistas del ejercicio 4.1</strong></summary>

- La actividad que representa retorno es `app_open`.
- Usa `COUNT(DISTINCT CASE WHEN ... THEN user_id END)`.
- Para semana 1, filtra `week_after_signup >= 1`.
- El denominador de cada tasa es `cohort_size`.
- Una cohorte reciente puede no haber tenido tiempo suficiente para llegar a la semana 3.

</details>

### Comparación conceptual

- El funnel detectó _________________________________________________.
- La retención permite evaluar ______________________________________.
- Una limitación temporal de las cohortes es _________________________.


## Errores frecuentes y cómo leerlos

| Señal | Causa probable | Acción |
|---|---|---|
| Conteos aumentan después | duplicación en el `JOIN` | revisar `DISTINCT` y llave |
| `column not found` | columna no seleccionada en la CTE | revisar la CTE anterior |
| División por cero | etapa sin usuarios | usar `NULLIF` |
| Todas las etapas iguales | filtro copiado | revisar cada `WHERE` |
| Se afirma causalidad | análisis descriptivo | redactar como hallazgo |


---
# Cierre · Validación de conocimiento

1. ¿Por qué comenzamos por el problema de negocio?
2. ¿Qué representa cada CTE?
3. ¿Por qué usamos `LEFT JOIN` desde `signup`?
4. ¿Qué diferencia hay entre conversión acumulada y entre etapas?
5. ¿Qué pregunta responde retención?
6. ¿Qué acción investigarías a partir del cuello de botella?

### Mis respuestas finales

1.  
2.  
3.  
4.  
5.  
6.  

**Tema que necesito repasar:** _________________________________________


## Takeaways

Completa cada idea con tus palabras:

- La granularidad es importante porque ________________________________.
- Una CTE por etapa ayuda a ___________________________________________.
- El denominador define ______________________________________________.
- Funnel y retención se diferencian en ________________________________.
- Mi principal conclusión de negocio es ______________________________.


### ¿Necesitas ayuda?

- `OFFICE HOURS`: dudas puntuales y revisión de consultas.
- `DA_CONSULTA`: preguntas sobre contenido o proyecto.
- Comparte siempre la consulta, el error y el resultado esperado.
